In [1]:
## REQUIRES conda_solar_wind enviroment from conda_solar_wind.yml

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import rioxarray
import xarray as xr
import numpy as np
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import os
import warnings
warnings.filterwarnings("ignore")

/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.31.0 or higher is recommended. You are running version 2.16.0
  warnings.warn(
ERROR 1: PROJ: proj_create_from_database: Open of /home/alessandrom/anaconda3/envs/bias_correction_conda/share/proj failed


In [2]:
# Methods that are used below.

def round_coords(xdf, digits = None):
    """
    Rounds the coordinates of the xarray DataArray or Dataset to the specified number of decimal places.
    Needed to prevent issues due to excess decimal digits in coordinates, leading to grid mismatches.

    Parameters:
    xdf (xarray.DataArray or xarray.Dataset): The data containing 'latitude' and 'longitude' to be rounded.
    digits (int, optional): The number of decimal places to round to. Defaults to 1 if not provided.

    Returns:
    xarray.DataArray or xarray.Dataset: The data with rounded coordinates.
    """
    if not digits:
        # Round latitude and longitude to 1 decimal place by default
        xdf['latitude'] = xdf.latitude.round(1).astype('float32')
        xdf['longitude'] = xdf.longitude.round(1).astype('float32')
    else:
        # Round latitude and longitude to the specified number of decimal places
        xdf['latitude'] = xdf.latitude.round(digits).astype('float32')
        xdf['longitude'] = xdf.longitude.round(digits).astype('float32')    
    return xdf

def make_dataset_era5land(target_var, orography, aspect, monthly_avg = False):
    """
    Creates a dataset by merging a target variable, orography, and aspect data, optionally computing monthly averages.

    Parameters:
    target_var (xarray.DataArray or xarray.Dataset): The target variable data.
    orography (xarray.DataArray or xarray.Dataset): Orography data.
    aspect (xarray.DataArray or xarray.Dataset): Aspect data.
    monthly_avg (bool, optional): Whether to compute monthly averages. Defaults to False.

    Returns:
    pandas.DataFrame: The merged dataset in a DataFrame format, with rounded coordinates.
    """
    target_var = target_var.interp_like(orography, method="nearest")

    for xdf in [target_var, orography, aspect]:
        xdf = round_coords(xdf)
    
    # Merge the datasets into one
    xdf_merged = xr.merge([orography, aspect, target_var])
    
    if monthly_avg:
        # Compute the monthly average if specified
        xdf_merged = xdf_merged.groupby(xdf_merged.time.dt.month).mean()
        
    # Convert to DataFrame and drop NaN values
    merged = xdf_merged.to_dataframe().dropna()
    merged = merged.reset_index()
    
    # Round coordinates for final output
    merged['longitude'] = merged['longitude'].round(1)
    merged['latitude'] = merged['latitude'].round(1)
    return merged

def make_dataset_glo90(orography, aspect):
    """
    Creates a dataset by merging GLO-90 orography and aspect data, with coordinates rounded to 6 decimal places (as 90 m correspond to about 0.000808°).

    Parameters:
    orography (xarray.DataArray or xarray.Dataset): Orography data.
    aspect (xarray.DataArray or xarray.Dataset): Aspect data.

    Returns:
    pandas.DataFrame: The merged dataset in a DataFrame format, with NaN values for zeroes and reset index.
    """
    for xdf in [orography, aspect]:
        xdf = round_coords(xdf, digits = 6)
        
    # Merge the GLO-90 datasets
    merged_glo90 = xr.merge([orography, aspect])
    
    # Convert to DataFrame and drop NaN values
    merged_glo90 = merged_glo90.to_dataframe().dropna()
    merged_glo90 = merged_glo90.reset_index()

    return merged_glo90

def era5scaler(xdf, features, target):
    """
    Scales the features and target variables of the ERA5Land dataset using StandardScaler.

    Parameters:
    xdf (pandas.DataFrame): The dataset containing the features and target.
    features (list of str): List of feature column names to scale.
    target (str): The target column name to scale.

    Returns:
    tuple: The scaler for the target, scaled feature array, and scaled target array.
    """
    featurescaler = StandardScaler()
    # Scale the features
    X_scaled = featurescaler.fit_transform(xdf[features])

    targetscaler = StandardScaler()
    # Scale the target
    y_scaled = targetscaler.fit_transform(xdf[[target]]).ravel()
    
    return targetscaler, X_scaled, y_scaled

def glo90scaler(xdf, features):
    """
    Scales the features of the GLO-90 dataset using StandardScaler.

    Parameters:
    xdf (pandas.DataFrame): The dataset containing the features.
    features (list of str): List of feature column names to scale.

    Returns:
    numpy.ndarray: The scaled feature array.
    """
    scaler_glo90 = StandardScaler()
    # Scale the features
    X_scaled = scaler_glo90.fit_transform(xdf[features])
    return X_scaled

In [3]:
def apply_downscaling(var, name, month):
    era5l_merged = make_dataset_era5land(var, orography, aspect, monthly_avg = True)
    era5l_merged = era5l_merged.loc[era5l_merged.month == month,:].drop('month', axis = 1)
    features = era5l_merged.columns.drop([name,'spatial_ref'])
    target = name
    targetscaler_era5, era5l_X, era5l_y = era5scaler(era5l_merged, features, target)
    regr = MLPRegressor(random_state=1, max_iter = 1000).fit(era5l_X, era5l_y)
    glo90_X = glo90scaler(dataset_glo90, features)
    downscaled_mlr_std = regr.predict(glo90_X) # These values are standardised, make them go back to actual values
    downscaled_mlr = targetscaler_era5.inverse_transform(downscaled_mlr_std.reshape(-1,1)) # Using the targetscaler
    return downscaled_mlr

In [4]:
country = 'croatia'

In [5]:
orography = xr.open_dataset(f'../DEMdata/{country}/era5land_orography_{country}.nc')
aspect = xr.open_dataset(f'../DEMdata/{country}/era5land_aspect_{country}.nc')
orography_glo90 = xr.open_dataset(f'../DEMdata/{country}/glo90_orography_{country}.nc')
aspect_glo90 = xr.open_dataset(f'../DEMdata/{country}/glo90_aspect_{country}.nc')
dataset_glo90 = make_dataset_glo90(orography_glo90,aspect_glo90)
dataset_glo90 = dataset_glo90.rename(columns = {'band_data':'aspect'})

In [8]:
exp = 'historical'          #ssp370              
model = 'CNRM-ESM2-1'

if exp == 'historical':
    start_date = '1985-01'
    end_date   = '2014-12'
else:
    start_date = '2015-01'
    end_date   = '2100-12'
rsds = xr.open_mfdataset(f'../data/processed_data/{country}/{exp}/{model}/rsds_{start_date}_{end_date}_processed.nc')

In [9]:
start_period = '1992' #2030 #SELECT THE STARTING YEAR OF THE DECADE I WANT TO ANALYZE
end_period   = '2020' #2049 #SELECT THE ENDING YEAR OF THE DECADE I WANT TO ANALYZE
rsds = rsds.sel(time=slice(f'{start_period}',f'{end_period}'))

The downscaling is applied to the mean along the selected months included between the start and the end years of the period.

In this way, it will be possible to compute the change in solar radiation between the future projection ad the reanalysis period. 

In [10]:
month = 1
var = 'rsds'
downscaled_mlr = apply_downscaling(rsds, var, month)
dataset_glo90['ssrd_downscaled'] = downscaled_mlr        
output = dataset_glo90.set_index(['latitude', 'longitude'])[['ssrd_downscaled']].to_xarray()      
output['ssrd_downscaled'] = output['ssrd_downscaled'] * 24 / 1000 # from W/m2 to kWh/m2/day      

output_path = f'../data/downscaled_data/{country}/{exp}/{model}/'
output_dir = os.path.dirname(output_path)
os.makedirs(output_dir, exist_ok=True)
output.to_netcdf(f'{output_path}ssrd_downscaled_{country}_m{month}_{start_period}_{end_period}.nc')